# U-Net++ implementation on 128x128 px tiles

## Setup and Imports

In [ ]:
# Install notebook dependencies in your active environment before running, for example:
# python -m pip install rasterio tensorflow matplotlib scikit-learn
        


In [ ]:
import os
import sys
import json
import subprocess
import numpy as np
import tensorflow as tf
import rasterio
import warnings
from pathlib import Path
from datetime import datetime
import time
import matplotlib.pyplot as plt
from sklearn.utils import shuffle
from tensorflow.keras import layers, models, callbacks, mixed_precision
from tensorflow.keras import backend as K

print(f"TensorFlow version: {tf.__version__}")
print(f"Python version: {sys.version}")


## GPU configuration

In [ ]:
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

        result = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total,memory.free",
                "--format=csv,noheader",
            ],
            capture_output=True,
            text=True,
            check=False,
        )
        if result.returncode == 0 and result.stdout.strip():
            print("GPU Information:")
            print(result.stdout.strip().splitlines()[0])
        else:
            print("GPU detected, but nvidia-smi is unavailable.")

        policy = mixed_precision.Policy("mixed_float16")
        mixed_precision.set_global_policy(policy)
        print(f"\nMixed precision policy: {policy.name}")
        print("Compute dtype:", policy.compute_dtype)
        print("Variable dtype:", policy.variable_dtype)

    except RuntimeError as e:
        print(f"GPU setup error: {e}")
else:
    print("No GPUs found")
        


## Pull data

In [ ]:
WORKING_DIR = Path.cwd().resolve()
DATA_DIR_ENV = "UNET_128_DATA_DIR"


def find_repo_root(start_path):
    for candidate in [start_path, *start_path.parents]:
        if (candidate / ".git").exists():
            return candidate
    return start_path


PROJECT_ROOT = find_repo_root(WORKING_DIR)
PAPER_REPRODUCTION_MODE = True
PAPER_BASE_PATH = PROJECT_ROOT / "spie" / "Preprocessed-128"

if not PAPER_BASE_PATH.exists():
    raise FileNotFoundError(
        f"Paper reproduction dataset not found: {PAPER_BASE_PATH}. "
        "Create Preprocessed-128-paper-filtered before running this notebook."
    )

BASE_PATH = PAPER_BASE_PATH
os.environ[DATA_DIR_ENV] = str(BASE_PATH)

TRAIN_SAR = BASE_PATH / "train" / "sar"
TRAIN_FLOOD = BASE_PATH / "train" / "flood"
VAL_SAR = BASE_PATH / "val" / "sar"
VAL_FLOOD = BASE_PATH / "val" / "flood"
TEST_SAR = BASE_PATH / "test" / "sar"
TEST_FLOOD = BASE_PATH / "test" / "flood"
METADATA_PATH = BASE_PATH / "metadata"

print(f"Working directory: {WORKING_DIR}")
print(f"Detected project root: {PROJECT_ROOT}")
print(f"Paper reproduction mode: {PAPER_REPRODUCTION_MODE}")
print(f"Using base dataset directory: {BASE_PATH}")

print("Checking data paths...")
for path_name, path in [
    ("Base", BASE_PATH),
    ("Train SAR", TRAIN_SAR),
    ("Train Flood", TRAIN_FLOOD),
    ("Val SAR", VAL_SAR),
    ("Val Flood", VAL_FLOOD),
    ("Test SAR", TEST_SAR),
    ("Test Flood", TEST_FLOOD),
    ("Metadata", METADATA_PATH),
]:
    if path.exists():
        if path.is_dir() and path_name not in {"Base", "Metadata"}:
            file_count = len(list(path.glob("*.tif")))
            print(f"{path_name}: {file_count} files")
        else:
            print(f"{path_name}: exists")
    else:
        print(f"{path_name}: not found at {path}")

norm_stats_path = METADATA_PATH / "normalization_stats.json"
if norm_stats_path.exists():
    with open(norm_stats_path, "r") as f:
        norm_stats = json.load(f)


## Data loading configuration

In [ ]:
IMG_HEIGHT = 128
IMG_WIDTH = 128
CHANNELS = 3
BATCH_SIZE = 8
EPOCHS = 20
LEARNING_RATE = 0.001
EVAL_THRESHOLD = 0.5
LOSS_NAME = "binary_crossentropy_plus_soft_dice_loss"
BUFFER_SIZE = 1000
AUTOTUNE = tf.data.AUTOTUNE

print(f"Image size: {IMG_HEIGHT}x{IMG_WIDTH}")
print(f"Channels: {CHANNELS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Adam learning rate: {LEARNING_RATE}")
print(f"Loss: {LOSS_NAME}")
print(f"Final evaluation threshold: {EVAL_THRESHOLD}")


## Data pipeline functions

In [ ]:
warnings.filterwarnings('ignore', category=rasterio.errors.NotGeoreferencedWarning)

def load_image_pair(sar_path, flood_path):

    with rasterio.open(sar_path.numpy().decode()) as src:
        sar_data = src.read().transpose(1, 2, 0).astype(np.float32)

    with rasterio.open(flood_path.numpy().decode()) as src:
        flood_data = src.read(1).astype(np.float32)
        flood_data = np.expand_dims(flood_data, axis=-1)

    return sar_data, flood_data

@tf.autograph.experimental.do_not_convert
def tf_load_image_pair(sar_path, flood_path):

    sar_data, flood_data = tf.py_function(
        load_image_pair,
        [sar_path, flood_path],
        [tf.float32, tf.float32]
    )

    sar_data.set_shape([IMG_HEIGHT, IMG_WIDTH, CHANNELS])
    flood_data.set_shape([IMG_HEIGHT, IMG_WIDTH, 1])

    return sar_data, flood_data

@tf.autograph.experimental.do_not_convert
def augment(sar, mask):

    # Keep SAR and mask transforms synchronized without relying on AutoGraph
    sar, mask = tf.cond(
        tf.random.uniform(()) > 0.5,
        lambda: (tf.image.flip_left_right(sar), tf.image.flip_left_right(mask)),
        lambda: (sar, mask),
    )

    sar, mask = tf.cond(
        tf.random.uniform(()) > 0.5,
        lambda: (tf.image.flip_up_down(sar), tf.image.flip_up_down(mask)),
        lambda: (sar, mask),
    )

    k = tf.random.uniform((), maxval=4, dtype=tf.int32)
    sar = tf.image.rot90(sar, k)
    mask = tf.image.rot90(mask, k)

    sar = tf.image.random_brightness(sar, 0.1)

    return sar, mask

def create_dataset(sar_dir, flood_dir, training=False, batch_size=BATCH_SIZE):

    sar_files = list(Path(sar_dir).glob('*.tif'))

    flood_map = {}
    for f in Path(flood_dir).glob('*.tif'):
        key = f.stem.replace('_flood_prep', '_prep')
        flood_map[key] = str(f)

    matched_pairs = []

    for sar_file in sar_files:
        sar_stem = sar_file.stem
        if sar_stem in flood_map:
            matched_pairs.append((str(sar_file), flood_map[sar_stem]))

    sar_paths = [pair[0] for pair in matched_pairs]
    flood_paths = [pair[1] for pair in matched_pairs]

    dataset = tf.data.Dataset.from_tensor_slices((sar_paths, flood_paths))

    if training:
        dataset = dataset.shuffle(buffer_size=BUFFER_SIZE, seed=42)

    dataset = dataset.map(tf_load_image_pair, num_parallel_calls=AUTOTUNE)

    if training:
        dataset = dataset.map(augment, num_parallel_calls=AUTOTUNE)

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset, len(matched_pairs)

## Create datasets

In [ ]:
train_dataset, train_size = create_dataset(TRAIN_SAR, TRAIN_FLOOD, training=True)
val_dataset, val_size = create_dataset(VAL_SAR, VAL_FLOOD, training=False)
test_dataset, test_size = create_dataset(TEST_SAR, TEST_FLOOD, training=False)

print(f"Train: {train_size} images ({train_size // BATCH_SIZE} batches)")
print(f"Validation: {val_size} images ({val_size // BATCH_SIZE} batches)")
print(f"Test: {test_size} images ({test_size // BATCH_SIZE} batches)")

for i, (sar, mask) in enumerate(train_dataset.take(1)):
    print(f"SAR batch shape: {sar.shape}, dtype: {sar.dtype}")
    print(f"Mask batch shape: {mask.shape}, dtype: {mask.dtype}")
    print(f"SAR value range: [{tf.reduce_min(sar):.3f}, {tf.reduce_max(sar):.3f}]")
    print(f"Mask unique values: {tf.unique(tf.reshape(mask, [-1]))[0].numpy()}")

## U-Net++ model architecture

In [ ]:
def conv_block(inputs, filters, kernel_size=3, dropout_rate=0.1):
    # Convolutional block with batch normalization and dropout
    x = layers.Conv2D(filters, kernel_size, padding='same', kernel_initializer='he_normal')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Conv2D(filters, kernel_size, padding='same', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    return x

def encoder_block(inputs, filters, pool_size=2, dropout_rate=0.1):
    # Encoder block with convolution and pooling
    conv = conv_block(inputs, filters, dropout_rate=dropout_rate)
    pool = layers.MaxPooling2D(pool_size)(conv)
    return conv, pool

def decoder_block(inputs, skip_features, filters, kernel_size=2, dropout_rate=0.1):
    # Decoder block with transpose convolution and skip connection
    x = layers.Conv2DTranspose(filters, kernel_size, strides=2, padding='same')(inputs)
    x = layers.concatenate([x, skip_features])
    x = conv_block(x, filters, dropout_rate=dropout_rate)
    return x

def build_unet_plus_plus(input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS), dropout_rate=0.1):
    inputs = layers.Input(shape=input_shape)

    # Encoder
    x00, p0 = encoder_block(inputs, 64, dropout_rate=dropout_rate)      # 128
    x10, p1 = encoder_block(p0, 128, dropout_rate=dropout_rate)         # 64
    x20, p2 = encoder_block(p1, 256, dropout_rate=dropout_rate)         # 32
    x30, p3 = encoder_block(p2, 512, dropout_rate=dropout_rate)         # 16

    # Bridge
    x40 = conv_block(p3, 1024, dropout_rate=dropout_rate)               # 8

    # Dense connections - Level 3
    x31 = decoder_block(x40, x30, 512, dropout_rate=dropout_rate)

    # Dense connections - Level 2
    x21 = decoder_block(x30, x20, 256, dropout_rate=dropout_rate)
    x22 = decoder_block(x31,
                       layers.concatenate([x20, x21]),
                       256, dropout_rate=dropout_rate)

    # Dense connections - Level 1
    x11 = decoder_block(x20, x10, 128, dropout_rate=dropout_rate)
    x12 = decoder_block(x21,
                       layers.concatenate([x10, x11]),
                       128, dropout_rate=dropout_rate)
    x13 = decoder_block(x22,
                       layers.concatenate([x10, x11, x12]),
                       128, dropout_rate=dropout_rate)

    # Dense connections - Level 0
    x01 = decoder_block(x10, x00, 64, dropout_rate=dropout_rate)
    x02 = decoder_block(x11,
                       layers.concatenate([x00, x01]),
                       64, dropout_rate=dropout_rate)
    x03 = decoder_block(x12,
                       layers.concatenate([x00, x01, x02]),
                       64, dropout_rate=dropout_rate)
    x04 = decoder_block(x13,
                       layers.concatenate([x00, x01, x02, x03]),
                       64, dropout_rate=dropout_rate)

    # Output
    outputs = layers.Conv2D(1, 1, activation='sigmoid', dtype='float32')(x04)

    model = models.Model(inputs=inputs, outputs=outputs)
    return model

model = build_unet_plus_plus(dropout_rate=0.1)
print(f"Total parameters: {model.count_params():,}")

## Loss Functions and Metrics

In [ ]:
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred)

    dice = (2.0 * intersection + smooth) / (union + smooth)
    return dice

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coefficient(y_true, y_pred)

def iou_score(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) - intersection

    iou = (intersection + smooth) / (union + smooth)
    return iou

def combined_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)
    return bce + dice

# Custom metrics
class SegmentationMetrics(tf.keras.metrics.Metric):
    def __init__(self, name='segmentation_metrics', **kwargs):
        super().__init__(name=name, **kwargs)
        self.true_positives = self.add_weight(name='tp', initializer='zeros')
        self.false_positives = self.add_weight(name='fp', initializer='zeros')
        self.false_negatives = self.add_weight(name='fn', initializer='zeros')
        self.true_negatives = self.add_weight(name='tn', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(y_true > 0.5, tf.float32)
        y_pred = tf.cast(y_pred > 0.5, tf.float32)

        self.true_positives.assign_add(tf.reduce_sum(y_true * y_pred))
        self.false_positives.assign_add(tf.reduce_sum((1 - y_true) * y_pred))
        self.false_negatives.assign_add(tf.reduce_sum(y_true * (1 - y_pred)))
        self.true_negatives.assign_add(tf.reduce_sum((1 - y_true) * (1 - y_pred)))

    def result(self):
        precision = self.true_positives / (self.true_positives + self.false_positives + K.epsilon())
        recall = self.true_positives / (self.true_positives + self.false_negatives + K.epsilon())
        return {'precision': precision, 'recall': recall}

    def reset_state(self):
        self.true_positives.assign(0.)
        self.false_positives.assign(0.)
        self.false_negatives.assign(0.)
        self.true_negatives.assign(0.)

## Compile model

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
optimizer = mixed_precision.LossScaleOptimizer(optimizer)

model.compile(
    optimizer=optimizer,
    loss=combined_loss,
    metrics=[dice_coefficient]
)

print("Model compiled successfully")
print("Training metric: soft Dice coefficient")


## Training configuration

In [ ]:
OUTPUT_DIR_ENV = "UNETPP_128_OUTPUT_DIR"

# Override with {OUTPUT_DIR_ENV} if you want checkpoints somewhere else.
checkpoint_dir = Path(
    os.environ.get(
        OUTPUT_DIR_ENV,
        str(Path.cwd() / "artifacts" / "paper_reproduction" / "unet++_128_checkpoints"),
    )
).expanduser()
checkpoint_dir.mkdir(parents=True, exist_ok=True)


class EpochTiming(callbacks.Callback):
    def __init__(self, samples_per_epoch):
        super().__init__()
        self.samples_per_epoch = samples_per_epoch
        self.train_start = None
        self.epoch_start = None

    def on_train_begin(self, logs=None):
        self.train_start = time.perf_counter()

    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_start = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        epoch_time = time.perf_counter() - self.epoch_start
        cumulative_time = time.perf_counter() - self.train_start
        logs["epoch_time_sec"] = epoch_time
        logs["cumulative_time_sec"] = cumulative_time
        logs["tiles_per_sec"] = self.samples_per_epoch / epoch_time if epoch_time > 0 else 0.0
        print(
            f"Epoch {epoch + 1} timing: "
            f"{epoch_time:.1f}s, {logs['tiles_per_sec']:.2f} tiles/sec, "
            f"cumulative {cumulative_time / 60:.1f} min"
        )

TRAIN_BATCHES = int(np.ceil(train_size / BATCH_SIZE))
VALIDATION_BATCHES = int(np.ceil(val_size / BATCH_SIZE))
TEST_BATCHES = int(np.ceil(test_size / BATCH_SIZE))

callbacks_list = [
    callbacks.ModelCheckpoint(
        filepath=str(checkpoint_dir / "best_model.h5"),
        monitor="val_dice_coefficient",
        mode="max",
        save_best_only=True,
        save_weights_only=False,
        verbose=1,
    ),
    EpochTiming(samples_per_epoch=train_size),
    callbacks.CSVLogger(
        str(checkpoint_dir / "training_log.csv"),
        append=False,
    ),
]

print()
print("Training configuration:")
print(f"Epochs: {EPOCHS}")
print(f"Train images: {train_size}")
print(f"Validation images: {val_size}")
print(f"Test images: {test_size}")
print(f"Train batches: {TRAIN_BATCHES}")
print(f"Validation batches: {VALIDATION_BATCHES}")
print(f"Test batches: {TEST_BATCHES}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Adam learning rate: {LEARNING_RATE}")
print("Callbacks: ModelCheckpoint, EpochTiming, CSVLogger")


## Train model

In [ ]:
print()
print(f"Starting training at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=val_dataset,
    callbacks=callbacks_list,
    verbose=1
)

print()
print(f"Training completed at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


## Plot training history

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Validation')
axes[0].set_title('BCE + Dice Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['dice_coefficient'], label='Train')
axes[1].plot(history.history['val_dice_coefficient'], label='Validation')
axes[1].set_title('Soft Dice Coefficient')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Soft Dice')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(checkpoint_dir / 'training_history.png', dpi=300, bbox_inches='tight')
plt.show()


## Evaluate model on test set

In [ ]:
best_model_path = checkpoint_dir / 'best_model.h5'
if best_model_path.exists():
    print("Loading best validation soft-Dice model...")

    model = build_unet_plus_plus(dropout_rate=0.1)

    optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    optimizer = mixed_precision.LossScaleOptimizer(optimizer)

    model.compile(
        optimizer=optimizer,
        loss=combined_loss,
        metrics=[dice_coefficient]
    )

    model.load_weights(best_model_path)
    print("Best model weights loaded successfully")
else:
    print("Using final model (best model checkpoint not found)")


def compute_thresholded_global_metrics(model, dataset, threshold=0.5):
    tp = 0
    fp = 0
    fn = 0
    tn = 0

    for sar_batch, mask_batch in dataset:
        pred_batch = model.predict(sar_batch, verbose=0)
        y_true = mask_batch.numpy() > 0.5
        y_pred = pred_batch > threshold

        tp += int(np.logical_and(y_true, y_pred).sum())
        fp += int(np.logical_and(~y_true, y_pred).sum())
        fn += int(np.logical_and(y_true, ~y_pred).sum())
        tn += int(np.logical_and(~y_true, ~y_pred).sum())

    total = tp + fp + fn + tn
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    pixel_accuracy = (tp + tn) / total if total > 0 else 0.0
    dice = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0

    return {
        'threshold': float(threshold),
        'true_positives': tp,
        'false_positives': fp,
        'false_negatives': fn,
        'true_negatives': tn,
        'total_pixels': total,
        'pixel_accuracy': float(pixel_accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'dice': float(dice),
    }

paper_style_metrics = compute_thresholded_global_metrics(
    model=model,
    dataset=test_dataset,
    threshold=EVAL_THRESHOLD,
)

print()
print(f"Paper-style thresholded global test metrics (threshold = {EVAL_THRESHOLD:.2f})")
print(f"Pixel Accuracy: {paper_style_metrics['pixel_accuracy']:.4f}")
print(f"Precision: {paper_style_metrics['precision']:.4f}")
print(f"Recall: {paper_style_metrics['recall']:.4f}")
print(f"Dice Coefficient: {paper_style_metrics['dice']:.4f}")
print(
    "Confusion counts: "
    f"TP={paper_style_metrics['true_positives']}, "
    f"FP={paper_style_metrics['false_positives']}, "
    f"FN={paper_style_metrics['false_negatives']}, "
    f"TN={paper_style_metrics['true_negatives']}"
)


## Save final results

In [ ]:
results_summary = {
    'run_completed': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'paper_reproduction_mode': PAPER_REPRODUCTION_MODE,
    'model': {
        'name': 'U-Net++',
        'architecture_changed': False,
        'parameters': int(model.count_params()),
        'dropout_rate': 0.1,
    },
    'dataset': {
        'base_path': str(BASE_PATH),
        'tile_size': [IMG_HEIGHT, IMG_WIDTH],
        'channels': CHANNELS,
        'train_samples': int(train_size),
        'validation_samples': int(val_size),
        'test_samples': int(test_size),
        'train_sar_dir': str(TRAIN_SAR),
        'train_flood_dir': str(TRAIN_FLOOD),
        'validation_sar_dir': str(VAL_SAR),
        'validation_flood_dir': str(VAL_FLOOD),
        'test_sar_dir': str(TEST_SAR),
        'test_flood_dir': str(TEST_FLOOD),
    },
    'training_config': {
        'batch_size': BATCH_SIZE,
        'epochs_configured': EPOCHS,
        'epochs_trained': len(history.history['loss']),
        'optimizer': 'Adam',
        'learning_rate': LEARNING_RATE,
        'loss': LOSS_NAME,
        'training_metric': 'soft_dice_coefficient',
        'train_batches': TRAIN_BATCHES,
        'validation_batches': VALIDATION_BATCHES,
        'test_batches': TEST_BATCHES,
        'callbacks': ['ModelCheckpoint', 'EpochTiming', 'CSVLogger'],
        'total_training_time_sec': float(sum(history.history.get('epoch_time_sec', []))),
        'mean_epoch_time_sec': float(np.mean(history.history.get('epoch_time_sec', [0]))),
        'mean_tiles_per_sec': float(np.mean(history.history.get('tiles_per_sec', [0]))),
    },
    'best_validation': {
        'soft_dice_coefficient': float(max(history.history['val_dice_coefficient'])),
        'loss': float(min(history.history['val_loss'])),
    },
    'final_test_metrics': {
        'definition': 'Global TP/FP/FN/TN accumulated over all test pixels after thresholding model probabilities at 0.5',
        'thresholded_global': paper_style_metrics,
    },
}

results_path = checkpoint_dir / 'training_results.json'
with open(results_path, 'w') as f:
    json.dump(results_summary, f, indent=2)

print()
print(f"Results saved to: {results_path}")
print("Training pipeline completed")


In [ ]:

import random
import matplotlib.patches as mpatches

def visualize_model_prediction_with_errors(model, test_sar_dir, test_flood_dir, save_path=None):
    """
    Visualize model prediction on a random test tile with error analysis
    Shows false positives (blue) and false negatives (red) overlaid on flood mask
    """

    # Get random test file
    sar_files = list(Path(test_sar_dir).glob('*.tif'))
    random_sar_file = random.choice(sar_files)

    # Find corresponding flood mask
    flood_map = {}
    for f in Path(test_flood_dir).glob('*.tif'):
        key = f.stem.replace('_flood_prep', '_prep')
        flood_map[key] = str(f)

    sar_stem = random_sar_file.stem
    if sar_stem not in flood_map:
        print(f"No corresponding flood mask found for {random_sar_file.name}")
        return

    flood_file = Path(flood_map[sar_stem])

    print(f"Selected UAVSAR tile: {random_sar_file.name}")
    print(f"Corresponding flood mask: {flood_file.name}")

    # Load the image pair
    with rasterio.open(random_sar_file) as src:
        sar_data = src.read().transpose(1, 2, 0).astype(np.float32)

    with rasterio.open(flood_file) as src:
        flood_mask = src.read(1).astype(np.float32)

    # Prepare data for model prediction
    sar_input = np.expand_dims(sar_data, axis=0)  # Add batch dimension

    # Make prediction
    prediction = model.predict(sar_input, verbose=0)[0]  # Remove batch dimension
    prediction = prediction.squeeze()  # Remove channel dimension if present

    # Convert to binary prediction (threshold at 0.5)
    pred_binary = (prediction > 0.5).astype(np.float32)
    true_binary = (flood_mask > 0.5).astype(np.float32)

    # Calculate error maps
    false_positives = (pred_binary == 1) & (true_binary == 0)  # Predicted water, actually land
    false_negatives = (pred_binary == 0) & (true_binary == 1)  # Predicted land, actually water
    true_positives = (pred_binary == 1) & (true_binary == 1)   # Correctly predicted water
    true_negatives = (pred_binary == 0) & (true_binary == 0)   # Correctly predicted land

    # Calculate metrics for this tile
    tp = np.sum(true_positives)
    fp = np.sum(false_positives)
    fn = np.sum(false_negatives)
    tn = np.sum(true_negatives)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    pixel_accuracy = (tp + tn) / (tp + fp + fn + tn)
    dice = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0

    print(f"\nTile Metrics:")
    print(f"Pixel Accuracy: {pixel_accuracy:.3f}")
    print(f"Dice Coefficient: {dice:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall: {recall:.3f}")
    print(f"False Positives: {fp} pixels")
    print(f"False Negatives: {fn} pixels")

    # Create visualization
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    # 1. Original UAVSAR tile
    if sar_data.shape[-1] == 3:
        # Handle potential negative values for display
        sar_display = sar_data.copy()
        # Normalize each channel for display
        for i in range(3):
            channel = sar_display[:, :, i]
            channel_min, channel_max = channel.min(), channel.max()
            if channel_max > channel_min:
                sar_display[:, :, i] = (channel - channel_min) / (channel_max - channel_min)
        axes[0].imshow(np.clip(sar_display, 0, 1))
    else:
        axes[0].imshow(sar_data.squeeze(), cmap='gray')

    axes[0].set_title('Original UAVSAR Tile\n(Pauli RGB)', fontsize=12, fontweight='bold')
    axes[0].axis('off')

    # 2. Ground truth flood mask
    axes[1].imshow(flood_mask, cmap='Blues', vmin=0, vmax=1)
    axes[1].set_title('Ground Truth\nFlood Mask', fontsize=12, fontweight='bold')
    axes[1].axis('off')

    # 3. Model prediction
    axes[2].imshow(prediction, cmap='Blues', vmin=0, vmax=1)
    axes[2].set_title(f'Model Prediction\n(Continuous)', fontsize=12, fontweight='bold')
    axes[2].axis('off')

    # 4. Error analysis overlay on flood mask
    # Start with black background
    error_overlay = np.zeros((flood_mask.shape[0], flood_mask.shape[1], 3))

    # True negatives: black (already set by zeros initialization)
    # True positives: white
    error_overlay[true_positives, 0] = 1.0  # Full red
    error_overlay[true_positives, 1] = 1.0  # Full green
    error_overlay[true_positives, 2] = 1.0  # Full blue

    # Add false positives in blue
    error_overlay[false_positives, 0] = 0.0  # No red
    error_overlay[false_positives, 1] = 0.0  # No green
    error_overlay[false_positives, 2] = 1.0  # Full blue

    # Add false negatives in red
    error_overlay[false_negatives, 0] = 1.0  # Full red
    error_overlay[false_negatives, 1] = 0.0  # No green
    error_overlay[false_negatives, 2] = 0.0  # No blue

    axes[3].imshow(error_overlay)
    axes[3].set_title('Error Analysis\n(Red=False Neg, Blue=False Pos)', fontsize=12, fontweight='bold')
    axes[3].axis('off')

    # Add legend for error analysis
    legend_elements = [
        mpatches.Patch(color='red', label=f'False Negatives ({fn} px)'),
        mpatches.Patch(color='blue', label=f'False Positives ({fp} px)'),
        mpatches.Patch(color='white', label=f'True Positives ({tp} px)'),
        mpatches.Patch(color='black', label=f'True Negatives ({tn} px)')
    ]

    fig.legend(handles=legend_elements, loc='lower center', ncol=4,
               bbox_to_anchor=(0.5, -0.05), fontsize=10)

    plt.suptitle(f'Model Prediction Analysis - {random_sar_file.name}',
                 fontsize=16, fontweight='bold', y=1.02)

    plt.tight_layout()

    # Save if path provided
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight',
                   facecolor='white', edgecolor='none')
        print(f"\nVisualization saved to: {save_path}")

    plt.show()

    return {
        'filename': random_sar_file.name,
        'metrics': {
            'pixel_accuracy': pixel_accuracy,
            'dice_coefficient': dice,
            'precision': precision,
            'recall': recall
        },
        'error_counts': {
            'false_positives': int(fp),
            'false_negatives': int(fn),
            'true_positives': int(tp),
            'true_negatives': int(tn)
        }
    }

# Run the visualization
print("Generating model prediction visualization...")
result = visualize_model_prediction_with_errors(
    model=model,
    test_sar_dir=TEST_SAR,
    test_flood_dir=TEST_FLOOD,
    save_path=checkpoint_dir / 'prediction_analysis.png'
)

print(f"\nAnalysis complete for tile: {result['filename']}")